***UCS-420 ASSIGNMENT 8***

**Name:** Kshitiz Poudel  
**Roll Number:** 1024170424

## Q1.
Write a unique paragraph (5-6 sentences) about your favorite topic (e.g., sports, technology, food, books, etc.).
1. Convert text to lowercase and remove punctuation using re.
2. Tokenize the text into words and sentences.
3. Split using split() and word_tokenize() and compare how Python split and NLTK’s word_tokenize() differ.
4. Remove stopwords (using NLTK's stopwords list).
5. Display word frequency distribution (excluding stopwords).

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter
import matplotlib.pyplot as plt

for package in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(package, quiet=True)


In [ ]:
text = """Space exploration helps us understand how the universe began and how planets change over time. Scientists use telescopes and robotic probes to study distant worlds before sending people there. Satellites also support everyday life by improving weather forecasts, maps, and communication. A future mission to Mars will need reliable machines, careful planning, and teamwork. I find space science exciting because each discovery leads to new questions. Learning about the stars reminds us that curiosity can take people far beyond Earth."""
print(text)

In [ ]:
# Convert the paragraph to lowercase and remove punctuation.
clean_text = re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())
print("Clean text:\n", clean_text)

sentences = sent_tokenize(text)
words = word_tokenize(clean_text)
print("\nSentences:", len(sentences))
print(sentences)
print("\nWord tokens:", len(words))
print(words)

split_tokens = text.lower().split()
nltk_tokens = word_tokenize(text.lower())
print("\nsplit() tokens:", split_tokens)
print("word_tokenize() tokens:", nltk_tokens)
print("Token count from split():", len(split_tokens))
print("Token count from word_tokenize():", len(nltk_tokens))
print("\nDifference in split() only:", sorted(set(split_tokens) - set(nltk_tokens)))
print("Difference in word_tokenize() only:", sorted(set(nltk_tokens) - set(split_tokens)))

stop_words = set(stopwords.words('english'))
filtered = [word for word in words if word.isalnum() and word not in stop_words]
print("\nTokens after removing stopwords:", filtered)

freq = Counter(filtered)
print("Word frequencies:", freq.most_common())
if freq:
    nltk.FreqDist(filtered).plot(len(freq), title='Word Frequency (without stopwords)')
    plt.show()

## Q2.
Using the same paragraph from Q1:
1. Extract all words with only alphabets using re.findall()
2. Remove stop words using NLTK’s stopword list
3. Perform stemming with PorterStemmer
4. Perform lemmatization with WordNetLemmatizer
5. Compare the stemmed and lemmatized outputs and explain when you’d prefer one over the other.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

alpha_words = re.findall(r'[a-zA-Z]+', text.lower())
print("Alphabet-only words:", alpha_words)

stop_words = set(stopwords.words('english'))
filtered_words = [word for word in alpha_words if word not in stop_words]
print("\nAfter stopword removal:", filtered_words)

porter = PorterStemmer()
lemma = WordNetLemmatizer()
stemmed = [porter.stem(word) for word in filtered_words]
lemmatized = [lemma.lemmatize(word) for word in filtered_words]

import pandas as pd
comparison = pd.DataFrame({
    'original': filtered_words,
    'stemmed': stemmed,
    'lemmatized': lemmatized
})
print("\nStemmed and lemmatized words:")
print(comparison.to_string(index=False))
print("\nRows where the outputs differ:")
print(comparison[comparison['stemmed'] != comparison['lemmatized']].to_string(index=False))

**Comparison:** Porter stemming removes word endings using rules, so its results can be shortened forms that are not real words. Lemmatization tries to return a dictionary form, which is usually easier to read. I would choose stemming when speed and matching related words are the main goals, such as a basic search system. I would prefer lemmatization when readable text and word meaning matter, for example in a text summary or a language application.

## Q3.
Choose 3 short texts of your own (e.g., different news headlines, product reviews).
1. Use CountVectorizer to generate the Bag of Words representation.
2. Use TfidfVectorizer to compute TF-IDF scores.
3. Print and interpret the top 3 keywords from each text using TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import numpy as np

texts = [
    "Community gardens give city residents fresh vegetables and a shared outdoor space.",
    "Electric buses reduce street pollution and make public transport quieter.",
    "Online learning platforms let students watch lessons and practise skills at their own pace."
]

labels = ['Garden text', 'Electric bus text', 'Online learning text']
count_vectorizer = CountVectorizer(stop_words='english')
bow = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(bow.toarray(), columns=count_vectorizer.get_feature_names_out(), index=labels)
print("Bag of Words counts:")
print(bow_df)

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out(), index=labels)
print("\nTF-IDF scores:")
print(tfidf_df.round(3))

terms = tfidf_vectorizer.get_feature_names_out()
for i, label in enumerate(labels):
    scores = tfidf_matrix[i].toarray().ravel()
    top_indices = np.argsort(scores)[::-1][:3]
    keywords = [(terms[j], round(float(scores[j]), 3)) for j in top_indices if scores[j] > 0]
    print(f"\nTop keywords for {label}: {keywords}")
    print("Interpretation: these words describe the main subject of this text and help distinguish it from the other two.")

## Q4.
Write 2 short texts (4–6 lines each) describing two different technologies (e.g., AI vs Blockchain).
1. Preprocess and tokenize both texts.
2. Calculate:
   a. Jaccard Similarity using sets
   b. Cosine Similarity using TfidfVectorizer + cosine_similarity()
   c. Analyze which similarity metric gives better insights in your case.

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

text_solar = """Solar panels convert sunlight into electricity for homes and businesses.
An inverter changes the generated power into a form household devices can use.
Battery storage can keep extra energy for the evening or cloudy weather.
Solar systems can lower electricity bills and reduce dependence on fossil fuels.
Their output depends on sunlight, panel position, and local weather."""

text_wind = """Wind turbines use moving air to generate electricity for communities and industry.
The blades turn a generator, producing power without burning fuel during operation.
Large wind farms can supply electricity to the grid across many regions.
Wind energy reduces dependence on fossil fuels, although output changes with weather.
Careful site selection helps turbines work efficiently and limits noise concerns."""

stop_words = set(stopwords.words('english'))
def preprocess(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    tokens = word_tokenize(text)
    return [token for token in tokens if token not in stop_words]

tokens_solar = preprocess(text_solar)
tokens_wind = preprocess(text_wind)
print("Solar tokens:", tokens_solar)
print("Wind tokens:", tokens_wind)

set_solar, set_wind = set(tokens_solar), set(tokens_wind)
union = set_solar | set_wind
jaccard = len(set_solar & set_wind) / len(union) if union else 0
print("\nShared words:", sorted(set_solar & set_wind))
print("Jaccard similarity:", round(jaccard, 4))

clean_solar, clean_wind = ' '.join(tokens_solar), ' '.join(tokens_wind)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([clean_solar, clean_wind])
cosine = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0, 0]
print("Cosine similarity:", round(float(cosine), 4))

**Analysis:** Jaccard similarity compares the sets of distinct words, so it is easy to explain but ignores how important each word is. TF-IDF cosine similarity compares weighted word vectors and gives less weight to words that appear frequently across the texts. Solar and wind energy have different equipment, but both descriptions discuss electricity, weather, and fossil fuels, so some overlap is expected. For these short technology descriptions, TF-IDF cosine similarity is more informative because it considers the distribution of terms. Neither metric fully understands meaning; both are based on word overlap.

## Q5.
Write a short review for a product or service.
1. Use TextBlob or VADER to find polarity & subjectivity for each review.
2. Classify reviews into Positive / Negative / Neutral.
3. Create a word cloud using the wordcloud library for all positive reviews.

In [ ]:
# If these libraries are missing in the current notebook environment, run:
# %pip install textblob wordcloud

from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd

reviews = [
    "The campus library is clean and quiet, and the staff help me find books quickly.",
    "The food delivery arrived cold and much later than the promised time.",
    "The mobile app is simple to use and the reminders are genuinely helpful.",
    "Customer support ignored my messages and did not solve the billing issue.",
    "The bus service is average; it gets me to campus but the seats are uncomfortable.",
    "The new study lamp gives bright light and makes late-night reading easier.",
    "The online booking page is confusing and it crashed twice while I paid."
]

rows = []
for review in reviews:
    sentiment = TextBlob(review).sentiment
    if sentiment.polarity > 0.1:
        category = 'Positive'
    elif sentiment.polarity < -0.1:
        category = 'Negative'
    else:
        category = 'Neutral'
    rows.append([review, round(sentiment.polarity, 3), round(sentiment.subjectivity, 3), category])

results = pd.DataFrame(rows, columns=['Review', 'Polarity', 'Subjectivity', 'Sentiment'])
print(results.to_string(index=False))
print("\nReview counts:")
print(results['Sentiment'].value_counts())

positive_reviews = ' '.join(results.loc[results['Sentiment'] == 'Positive', 'Review'])
if positive_reviews.strip():
    cloud = WordCloud(width=900, height=450, background_color='white',
                      stopwords=set(stopwords.words('english'))).generate(positive_reviews)
    plt.figure(figsize=(11, 5))
    plt.imshow(cloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Positive Reviews')
    plt.show()
else:
    print("No positive reviews were classified, so a word cloud could not be created.")

## Q6.
Choose your own paragraph (~100 words) as training data.
1. Tokenize text using Tokenizer() from keras.preprocessing.text
2. Create input sequences and build a simple LSTM or Dense model
3. Train the model and generate 2–3 new lines of text starting from any seed word you provide.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

# A short original training paragraph about learning to play the piano.
training_text = """Learning piano takes patience and regular practice. A student begins with simple notes and slowly learns to read music. Daily practice helps the fingers move with more control. Listening carefully improves rhythm and makes each melody clearer. Some pieces feel difficult at first, but small goals make progress easier to notice. A good teacher gives useful feedback and encourages steady effort. With time, practice becomes a habit and playing music becomes more enjoyable. Music can express feelings that are difficult to explain with words. Every new song gives the learner another chance to listen, practise, and improve."""

# Tokenize the training text and build next-word training sequences.
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts([training_text])
sequences = []
for line in training_text.split('. '):
    token_ids = tokenizer.texts_to_sequences([line])[0]
    for i in range(2, len(token_ids) + 1):
        sequences.append(token_ids[:i])

if not sequences:
    raise ValueError('Not enough words in the training text to create sequences.')

max_sequence_len = max(len(seq) for seq in sequences)
padded = pad_sequences(sequences, maxlen=max_sequence_len, padding='pre')
X = padded[:, :-1]
y = padded[:, -1]
vocab_size = len(tokenizer.word_index) + 1

print('Vocabulary size:', vocab_size)
print('Number of training sequences:', len(sequences))
print('Input shape:', X.shape)

# A small Dense next-word model.
tf.random.set_seed(24)
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16, input_length=X.shape[1]),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=0)
print('Training complete.')

# Generate three short lines from the seed word 'music'.
def generate_line(seed, words_to_add=7):
    result = seed.lower().strip()
    for _ in range(words_to_add):
        ids = tokenizer.texts_to_sequences([result])[0]
        ids = ids[-(max_sequence_len - 1):]
        padded_seed = pad_sequences([ids], maxlen=X.shape[1], padding='pre')
        probabilities = model.predict(padded_seed, verbose=0)[0]
        probabilities[0] = 0
        next_id = int(np.argmax(probabilities))
        if next_id == 0:
            break
        next_word = tokenizer.index_word.get(next_id, '')
        if not next_word or next_word == '<OOV>':
            break
        result += ' ' + next_word
    return result.capitalize() + '.'

seed_word = 'music'
print('Seed word:', seed_word)
for _ in range(3):
    print(generate_line(seed_word))